# Baseline: Engineered Features + Gradient Boosting

Before reaching for deep learning, we establish a classical baseline: hand-crafted features that encode the *physics-aware* description of a wafer map, fed to a gradient-boosted tree classifier.

This matters for two reasons:
1. **It sets the bar.** If a CNN can't beat interpretable features, it isn't earning its complexity.
2. **The features are explainable to a fab audience** — "high radial density in the outer ring" is a sentence a defect engineer can act on; a conv filter activation is not.

Feature groups (see `src/features.py`):
- **Radial ring densities** — fail rate in 5 concentric rings (separates Center / Donut / Edge-Ring)
- **Quadrant densities** — coarse location (Loc / Edge-Loc asymmetries)
- **Projection (radon-style) statistics** — directional structure (Scratch has one high-variance projection angle)
- **Region geometry** — connected-component count, dominance and elongation of the largest failing region (cluster vs. random vs. scratch)

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from src import data
from src.features import extract_features, FEATURE_NAMES

sns.set_theme(style="whitegrid")
X_maps, y = data.load_labeled()
print(X_maps.shape, y.shape)

### Subsample the majority class

"none" dwarfs everything else and adds little signal beyond a point. We keep **all** pattern wafers and cap "none" at 20,000 — feature extraction cost then stays in minutes while every rare class keeps its full data.

In [ ]:
rng = np.random.default_rng(42)
none_idx = data.CLASSES.index("none")
keep = np.flatnonzero(y != none_idx)
none_pool = np.flatnonzero(y == none_idx)
keep = np.concatenate([keep, rng.choice(none_pool, size=min(20_000, len(none_pool)), replace=False)])
Xs, ys = X_maps[keep], y[keep]
print(f"Subsample: {len(ys):,} maps")

feats = Parallel(n_jobs=-1, batch_size=256)(delayed(extract_features)(m) for m in Xs)
F = np.stack(feats)
print(f"Feature matrix: {F.shape}")

In [ ]:
F_tr, F_te, y_tr, y_te = train_test_split(F, ys, test_size=0.2, stratify=ys, random_state=42)

clf = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
clf.fit(F_tr, y_tr)
pred = clf.predict(F_te)
print(classification_report(y_te, pred, target_names=data.CLASSES, digits=3))

In [ ]:
cm = confusion_matrix(y_te, pred, normalize="true")
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=data.CLASSES, yticklabels=data.CLASSES, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Baseline confusion matrix (row-normalized)")
plt.tight_layout()
plt.savefig("../reports/figures/baseline_confusion.png", dpi=150)
plt.show()

### Which features carry the signal?

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(clf, F_te, y_te, n_repeats=5, random_state=42, n_jobs=-1)
order = imp.importances_mean.argsort()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(np.array(FEATURE_NAMES)[order], imp.importances_mean[order])
ax.set_title("Permutation importance (test set)")
plt.tight_layout()
plt.savefig("../reports/figures/feature_importance.png", dpi=150)
plt.show()

## Takeaways

- Interpretable features already separate the geometrically distinctive classes (Edge-Ring, Center, Near-full) well.
- The hard confusions are the *diffuse* ones — Loc vs. Edge-Loc vs. Random — where a fixed feature recipe loses spatial detail. That's exactly where a CNN should help (`03_cnn.ipynb`).
- In production, per-class **recall on rare actionable classes** is the metric that matters: a missed Donut is a missed excursion signal, while a false "none" flag just costs a review click.